# DGA Detection - Refactored
This notebook demonstrates the DGA detection pipeline using the refactored code in the `src/` directory.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
from sklearn.metrics import average_precision_score

# Add root directory to path to import src
sys.path.append(os.path.abspath('..'))

from src.data import read_data, remove_TLD, input_malignant, concatenate_all
from src.features import prepare_ensemble_features
from src.models import train_catboost, save_model

## 1. Data Loading
We use the functions from `src/data.py` to handle raw data. For this example, we assume concatenated data is available, as expected by `train.py`.

In [ ]:
# Configuration
input_data = '../data/training_data/data.csv'
ensemble_features_dir = '../data/training_data/ensemble_features'
max_len = 26
nrows = 1000

print(f"Loading data from {input_data} (nrows={nrows})...")
try:
    df = pd.read_csv(input_data, nrows=nrows)
except FileNotFoundError:
    print("Data file not found, creating dummy data for demonstration...")
    df = pd.DataFrame({'domain': ['google.com', 'bad-domain123.com', 'example.com', 'dga-gen-123.com'], 
                       'label': ['bening', 'dga', 'bening', 'dga']})
df.head()

## 2. Feature Extraction
Using `src/features.py` to extract DistilBERT tokens and byte-level features.

In [ ]:
print("Preparing features...")
df_res = prepare_ensemble_features(
    df, out_dir=ensemble_features_dir if os.path.exists(ensemble_features_dir) else None, 
    max_len=max_len, nrows=nrows, save=False
)

def cast_to_array(s):
    if isinstance(s, list):
        return np.array(s)
    if isinstance(s, np.ndarray):
        return s
    return np.array([int(el) for el in str(s)[1:-1].split(", ") if el])

df_res['features'] = df_res['features'].apply(cast_to_array)
print("Features prepared.")
df_res.head()

## 3. Train/Dev/Test Split

In [ ]:
# Check dataset size and split
size = df_res.shape[0]
train_size, dev_size, test_size = int(size * 0.8), int(size * 0.9), size
print(f"train: {train_size:,}, dev: {dev_size - train_size:,}, test: {test_size - dev_size:,}")

X_train = np.array(list(df_res.loc[0:train_size, 'features']))
y_train = df_res.loc[0:train_size, 'y']

X_dev = np.array(list(df_res.loc[train_size:dev_size, 'features']))
y_dev = df_res.loc[train_size:dev_size, 'y']

X_test = np.array(list(df_res.loc[dev_size:, 'features']))
y_test = df_res.loc[dev_size:, 'y']

## 4. Modeling
Training a CatBoost classifier utilizing `src/models.py`.

In [ ]:
print("Start training...")
if len(X_train) > 0:
    model = train_catboost(X_train, y_train, X_dev, y_dev, loss_function='MultiClass')
    print("Training complete.")
else:
    print("Not enough data to train.")

## 5. Evaluation & Saving

In [ ]:
print("Evaluating model...")
if len(X_test) > 0 and 'model' in locals():
    y_pred_proba = model.predict_proba(X_test)
    y_scores = [p[1] for p in y_pred_proba]
    
    # Needs at least one positive class for AP score calculation to make sense
    if sum(y_test) > 0:
        aps = average_precision_score(y_test, y_scores)
        print(f"Average Precision Score: {aps:.3f}")
    else:
        print("Test set contains only one class, skipping AP score.")
    
    save_model(model, '../models/catboost_ensemble_refactored.model')
else:
    print("Evaluation skipped due to lack of test data.")